# Race Position Prediction

**FIXES:**
1. ✅ Deduplicates properly
2. ✅ Adds circuit features

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_palette("husl")
%matplotlib inline

## 1. Load & Clean Data

In [ ]:
df = pd.read_parquet('../data/features/ml_features_2022_2025.parquet')

print(f"Loaded: {df.shape[0]:,} rows")
print(f"Years: {df['year'].min()}-{df['year'].max()}")
print(f"\nColumns available:")
print([c for c in df.columns if 'corner' in c or 'speed' in c or 'chicane' in c])

📊 Loaded: 16,924 rows
   Years: 2022-2025

Columns available:
['slow_corner_pct', 'medium_corner_pct', 'fast_corner_pct', 'total_slow_corners', 'total_medium_corners', 'total_fast_corners', 'chicanes', 'avg_speed_circuit', 'top_speed_circuit']


In [ ]:
# Filter to races only
df_race = df[
    (df['qualifying_position'].notna()) & 
    (df['race_position'].notna())
].copy()

print(f"Initial (with duplicates): {len(df_race):,} rows")

# FIX: DEDUPLICATE - keep first occurrence
print(f"\n Removing duplicates...")
before = len(df_race)

# FIX: Removing rookies - drivers with at least some history
print(f"\n Removing rookies...")

min_history_races = 3

# Check driver history completeness
driver_completeness = df_race.groupby('driver').agg({
    'circuit_avg_position': lambda x: x.notna().sum(),
    'driver_avg_position_change': lambda x: x.notna().sum()
})

# Keep only drivers with minimum history
valid_drivers = driver_completeness[
    (driver_completeness['circuit_avg_position'] >= min_history_races) &
    (driver_completeness['driver_avg_position_change'] >= min_history_races)
].index

df_race = df_race[df_race['driver'].isin(valid_drivers)]

print(f"Filtered to {len(df_race)} samples with complete driver history")

# Simple deduplication by year-event-driver
df_race = df_race.drop_duplicates(subset=['year', 'event', 'driver'], keep='first')

after = len(df_race)
removed = before - after
print(f"Removed: {removed:,} duplicates ({100*removed/before:.1f}%)")
print(f"Kept:    {after:,} unique driver-races")

# Stats
print(f"\n✅ Clean dataset:")
print(f"Races: {df_race.groupby(['year', 'event']).ngroups}")
print(f"Drivers: {df_race['driver'].nunique()}")

df_race['position_change'] = df_race['race_position'] - df_race['qualifying_position']
print(f"\n Avg position change: {df_race['position_change'].mean():.2f} ± {df_race['position_change'].std():.2f}")

Initial (with duplicates): 16,500 rows

🔧 Removing duplicates...

🔧 Removing rookies...
Filtered to 15107 samples with complete driver history
   Removed: 14,868 duplicates (90.1%)
   Kept:    1,632 unique driver-races

✅ Clean dataset:
   Races: 89
   Drivers: 23

📈 Avg position change: 0.04 ± 4.83


## 2. Feature Selection

**Including circuit characteristics!**

In [ ]:
desired_features = [
    # Starting position (CRITICAL)
    'qualifying_position',
    
    # Historical
    'circuit_avg_position',
    'circuit_best_position',
    'recent_avg_position',
    'recent_best_position',
    'form_trend',
    
    # Team
    'team_circuit_avg_position',
    'team_momentum',
    
    # Weather
    'avg_rainfall',
    'wet_dry_delta',
    
    # Telemetry
    'max_throttle_ratio',
    
    # CIRCUIT FEATURES (for overtaking prediction)
    'slow_corner_pct',       # Renamed from low_pct
    'medium_corner_pct',     # Renamed from med_pct
    'fast_corner_pct',       # Renamed from high_pct
    'total_slow_corners',    # From slow_corners
    'total_medium_corners',  # From medium_corners
    'total_fast_corners',    # From fast_corners
    'chicanes',
    'avg_speed_circuit',     # Renamed from avg_speed
    'top_speed_circuit',     # Renamed from top_speed

    # Circuit overtaking features
    'circuit_avg_position_change',      # Average position change at circuit
    'circuit_std_position_change',      # Variability
    'circuit_abs_position_change',      # Overtaking difficulty (KEY!)
    'circuit_max_gain',                 # Biggest historical gain
    'circuit_max_loss',                 # Biggest historical loss
    
    # Driver overtaking skill
    'driver_avg_position_change',       # Driver's overtaking tendency
    'driver_overtaking_success_rate',   # % races gained positions
    'driver_defensive_success_rate',    # % races defended well
]

available = [f for f in desired_features if f in df_race.columns]

print(f"✅ Using {len(available)}/{len(desired_features)} features:\n")
for i, feat in enumerate(available, 1):
    missing = (df_race[feat].isnull().sum() / len(df_race)) * 100
    print(f"{i:2d}. {feat:30s} (missing: {missing:.1f}%)")

missing = [f for f in desired_features if f not in df_race.columns]
if missing:
    print(f"\n⚠️  Not available ({len(missing)}):")
    for f in missing:
        print(f"- {f}")

✅ Using 28/28 features:

   1. qualifying_position            (missing: 0.0%)
   2. circuit_avg_position           (missing: 34.3%)
   3. circuit_best_position          (missing: 34.3%)
   4. recent_avg_position            (missing: 26.7%)
   5. recent_best_position           (missing: 26.7%)
   6. form_trend                     (missing: 26.7%)
   7. team_circuit_avg_position      (missing: 33.6%)
   8. team_momentum                  (missing: 4.5%)
   9. avg_rainfall                   (missing: 0.0%)
  10. wet_dry_delta                  (missing: 0.1%)
  11. max_throttle_ratio             (missing: 0.0%)
  12. slow_corner_pct                (missing: 6.1%)
  13. medium_corner_pct              (missing: 6.1%)
  14. fast_corner_pct                (missing: 6.1%)
  15. total_slow_corners             (missing: 6.1%)
  16. total_medium_corners           (missing: 6.1%)
  17. total_fast_corners             (missing: 6.1%)
  18. chicanes                       (missing: 6.1%)
  19. avg_speed

In [ ]:
# Verify qualifying_position exists
if 'qualifying_position' not in available:
    raise ValueError("❌ qualifying_position is MISSING!")
print("✅ qualifying_position confirmed")

✅ qualifying_position confirmed


In [ ]:
# Prepare modeling dataset
meta = ['year', 'event', 'driver', 'team']
meta = [c for c in meta if c in df_race.columns]

df_model = df_race[available + meta + ['race_position']].copy()

# Impute
print("Imputing missing values...")
imputed = []
for col in available:
    missing = df_model[col].isnull().sum()
    if missing > 0:
        median = df_model[col].median()
        df_model[col].fillna(median, inplace=True)
        imputed.append(f"{col} ({missing})")

if imputed:
    print(f"Imputed {len(imputed)} features")
else:
    print(f"No imputation needed")

print(f"\n✅ Ready: {df_model.shape}")
print(f"Features: {len(available)}")
print(f"Samples: {len(df_model):,}")
print(f"Missing: {df_model[available].isnull().sum().sum()}")

🔧 Imputing missing values...
   Imputed 25 features

✅ Ready: (1632, 33)
   Features: 28
   Samples: 1,632
   Missing: 0


## 3. Train/Test Split

In [ ]:
train = df_model[df_model['year'] < 2024].copy()
test = df_model[df_model['year'] == 2024].copy()

print(f"Split:")
print(f"Train: {len(train):,} samples ({train['year'].min()}-{train['year'].max()})")
print(f"Test:  {len(test):,} samples ({test['year'].unique()[0]})")

X_train = train[available].values
y_train = train['race_position'].values
X_test = test[available].values
y_test = test['race_position'].values

print(f"\n✅ X_train: {X_train.shape}, X_test: {X_test.shape}")

📊 Split:
   Train: 804 samples (2022-2023)
   Test:  478 samples (2024)

✅ X_train: (804, 28), X_test: (478, 28)


## 4. Baseline

In [ ]:
# Naive: race = qualifying
y_baseline = test['qualifying_position'].values

baseline_mae = mean_absolute_error(y_test, y_baseline)
baseline_r2 = r2_score(y_test, y_baseline)

print("BASELINE (race = qualifying)")
print(f"MAE: {baseline_mae:.3f} positions")
print(f"R²:  {baseline_r2:.3f}")

BASELINE (race = qualifying)
MAE: 2.822 positions
R²:  0.500


## 5. Train Random Forest

In [ ]:
print("🌲 Training...\n")

rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RANDOM FOREST")
print(f"MAE:  {mae:.3f} positions")
print(f"R²:   {r2:.3f}")

imp = baseline_mae - mae
imp_pct = (imp / baseline_mae) * 100

print(f"\nVs Baseline:")
print(f"Baseline: {baseline_mae:.3f}")
print(f"Model:    {mae:.3f}")
print(f"Diff:     {imp:+.3f} ({imp_pct:+.1f}%)")

if imp > 0:
    print(f"\n✅ BEATS baseline by {imp:.3f} positions!")
else:
    print(f"\n❌ WORSE than baseline by {abs(imp):.3f} positions")


🌲 Training...

RANDOM FOREST
MAE:  3.255 positions
R²:   0.491

Vs Baseline:
  Baseline: 2.822
  Model:    3.255
  Diff:     -0.433 (-15.4%)

❌ WORSE than baseline by 0.433 positions


## 6. Feature Importance

In [ ]:
importance = pd.DataFrame({
    'feature': available,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Features:\n")
for i, (_, row) in enumerate(importance.head(10).iterrows(), 1):
    bar = '█' * int(row['importance'] * 100)
    print(f"{i:2d}. {row['feature']:30s} {row['importance']:.3f} {bar}")

# Check if circuit features are important
circuit_feats = [f for f in available if 'corner' in f or 'speed' in f or 'chicane' in f]
if circuit_feats:
    circuit_in_top5 = any(f in importance['feature'].head(5).values for f in circuit_feats)
    if circuit_in_top5:
        print(f"\n✅ Circuit features in top 5!")
    else:
        print(f"\n⚠️  Circuit features not in top 5")

📊 Top 10 Features:

 1. qualifying_position            0.477 ███████████████████████████████████████████████
 2. recent_avg_position            0.079 ███████
 3. max_throttle_ratio             0.064 ██████
 4. wet_dry_delta                  0.061 ██████
 5. team_momentum                  0.055 █████
 6. medium_corner_pct              0.021 ██
 7. slow_corner_pct                0.020 ██
 8. team_circuit_avg_position      0.020 █
 9. top_speed_circuit              0.019 █
10. total_medium_corners           0.015 █

⚠️  Circuit features not in top 5


In [ ]:
# Plot
px.bar(
    importance.sort_values('importance', ascending=True).tail(10),
    y='feature',
    x='importance',
    title='Top 10 Features for Race Prediction',
    labels={'importance': 'Importance', 'feature': 'Feature'},
    height=400
).show()

## 7. Error Analysis

In [ ]:
analysis = test.copy()
analysis['predicted'] = y_pred
analysis['error'] = y_pred - y_test
analysis['abs_error'] = np.abs(analysis['error'])

print("❌ Worst 10 Predictions:\n")
worst = analysis.nlargest(10, 'abs_error')[[
    'event', 'driver', 'qualifying_position', 'race_position', 'predicted', 'abs_error'
]]

for _, row in worst.iterrows():
    print(f"{row['event']:30s} {row['driver']:3s} Q:{row['qualifying_position']:2.0f} "
          f"R:{row['race_position']:2.0f} P:{row['predicted']:4.1f} E:{row['abs_error']:4.1f}")

❌ Worst 10 Predictions:

Austrian Grand Prix            NOR Q: 2 R:20 P: 4.8 E:15.2
British Grand Prix             RUS Q: 1 R:19 P: 4.3 E:14.7
Australian Grand Prix          VER Q: 1 R:19 P: 4.9 E:14.1
Las Vegas Grand Prix           GAS Q: 3 R:20 P: 6.0 E:14.0
Belgian Grand Prix             RUS Q: 7 R:20 P: 8.0 E:12.0
Azerbaijan Grand Prix          SAI Q: 3 R:18 P: 6.4 E:11.6
Azerbaijan Grand Prix          PER Q: 4 R:17 P: 6.1 E:10.9
Canadian Grand Prix            LEC Q:11 R:19 P: 8.2 E:10.8
Abu Dhabi Grand Prix           PER Q:10 R:20 P: 9.7 E:10.3
United States Grand Prix       HAM Q:19 R:20 P: 9.9 E:10.1


In [ ]:
# Check for duplicates in output
dup = analysis.groupby(['event', 'driver']).size()
dups = dup[dup > 1]

if len(dups) > 0:
    print(f"\n❌ WARNING: Still {len(dups)} duplicates in output!")
    print(dups.head())
else:
    print(f"\n✅ No duplicates in predictions!")


✅ No duplicates in predictions!


## 8. Summary

In [ ]:
print("FINAL SUMMARY")
print(f"\nDataset:")
print(f"Train: {len(train):,} samples (2022-2023)")
print(f"Test:  {len(test):,} samples (2024)")

print(f"\nModel:")
print(f"Features: {len(available)}")
print(f"Top 3: {', '.join(importance['feature'].head(3).tolist())}")

print(f"\nPerformance:")
print(f"Baseline MAE: {baseline_mae:.3f}")
print(f"Model MAE:    {mae:.3f}")
print(f"Improvement:  {imp:+.3f} ({imp_pct:+.1f}%)")
print(f"R²:           {r2:.3f}")

print(f"\nInsights:")
if 'qualifying_position' in importance['feature'].head(1).values:
    print(f"✅ Starting position is #1 feature (correct)")

circuit_feats = [f for f in available if 'corner' in f or 'speed' in f]
if any(f in importance['feature'].head(5).values for f in circuit_feats):
    print(f"✅ Circuit features in top 5 (learning overtaking!)")

if imp > 0:
    print(f"✅ Model beats baseline")
else:
    print(f"❌ Model underperforms (features may not help)")


FINAL SUMMARY

Dataset:
  Train: 804 samples (2022-2023)
  Test:  478 samples (2024)

Model:
  Features: 28
  Top 3: qualifying_position, recent_avg_position, max_throttle_ratio

Performance:
  Baseline MAE: 2.822
  Model MAE:    3.255
  Improvement:  -0.433 (-15.4%)
  R²:           0.491

Insights:
  ✅ Starting position is #1 feature (correct)
  ❌ Model underperforms (features may not help)



## 9. NEW APPROACH: Predict Position CHANGE

In [ ]:

overtaking_feats = [
    'circuit_abs_position_change',
    'circuit_avg_position_change', 
    'driver_avg_position_change'
]

for feat in overtaking_feats:
    if feat in df_race.columns:
        missing_pct = df_race[feat].isnull().sum() / len(df_race) * 100
        print(f"{feat}: {missing_pct:.1f}% missing")
        
        # Check values
        print(f"Range: {df_race[feat].min():.2f} to {df_race[feat].max():.2f}")
        print(f"Std: {df_race[feat].std():.2f}\n")

circuit_abs_position_change: 26.7% missing
  Range: 2.10 to 5.30
  Std: 0.77

circuit_avg_position_change: 26.7% missing
  Range: -0.32 to 0.17
  Std: 0.06

driver_avg_position_change: 28.1% missing
  Range: -3.73 to 4.05
  Std: 1.53



In [ ]:

# 1. Create target = position change
df_race['position_change'] = df_race['race_position'] - df_race['qualifying_position']

# 2. Use ONLY high-quality features (low missing %)
simple_features = [
    'qualifying_position',  # Starting position matters for overtaking
    'slow_corner_pct',      # Circuit type
    'medium_corner_pct',
    'fast_corner_pct',
    'avg_speed_circuit',
    'max_throttle_ratio',
    'wet_dry_delta',
]

# 3. Filter to ONLY complete cases
df_complete = df_race[
    simple_features + ['position_change', 'race_position', 'year', 'event', 'driver']
].dropna()  

print(f"Complete cases: {len(df_complete)} (from {len(df_race)})")
print(f"Features: {len(simple_features)}")
print(f"Missing data: {df_complete[simple_features].isnull().sum().sum()}")

# 4. Train/test split
train = df_complete[df_complete['year'] < 2024]
test = df_complete[df_complete['year'] == 2024]

X_train = train[simple_features].values
y_train = train['position_change'].values  # ← PREDICT CHANGE!

X_test = test[simple_features].values
y_test = test['position_change'].values

# 5. Baseline: No position change
y_baseline = np.zeros(len(y_test))  # Assume no overtaking
baseline_mae = mean_absolute_error(y_test, y_baseline)

print(f"\nBaseline (no change): MAE = {baseline_mae:.3f} positions")

# 6. Train simpler model (Linear Regression!)
from sklearn.linear_model import Ridge

lr = Ridge(alpha=1.0)
lr.fit(X_train, y_train)
y_pred_change = lr.predict(X_test)

mae_change = mean_absolute_error(y_test, y_pred_change)

print(f"Model (Ridge):        MAE = {mae_change:.3f} positions")
print(f"Improvement: {baseline_mae - mae_change:.3f}")

# 7. Convert back to absolute positions
y_pred_race = test['qualifying_position'].values + y_pred_change
y_actual_race = test['race_position'].values

final_mae = mean_absolute_error(y_actual_race, y_pred_race)
baseline_race_mae = mean_absolute_error(
    y_actual_race, 
    test['qualifying_position'].values
)

print(f"\n==== FINAL RACE POSITION PREDICTION ====")
print(f"Baseline (race=quali): {baseline_race_mae:.3f}")
print(f"Model:                 {final_mae:.3f}")
print(f"Improvement:           {baseline_race_mae - final_mae:+.3f}")

# 8. Feature importance (Linear model coefficients)
importance = pd.DataFrame({
    'feature': simple_features,
    'coefficient': lr.coef_
}).sort_values('coefficient', key=abs, ascending=False)

print("\nFeature Importance (Ridge):")
for _, row in importance.iterrows():
    print(f"{row['feature']:25s} {row['coefficient']:+.3f}")

Complete cases: 1530 (from 1632)
Features: 7
Missing data: 0

Baseline (no change): MAE = 2.822 positions
Model (Ridge):        MAE = 3.009 positions
Improvement: -0.187

==== FINAL RACE POSITION PREDICTION ====
Baseline (race=quali): 2.822
Model:                 3.009
Improvement:           -0.187

Feature Importance (Ridge):
  max_throttle_ratio        -1.911
  qualifying_position       -0.402
  wet_dry_delta             -0.116
  slow_corner_pct           +0.068
  fast_corner_pct           -0.040
  medium_corner_pct         -0.028
  avg_speed_circuit         +0.006


# Current models and features do not allow proper race position prediction